In [1]:
import sys
sys.path.append('../')

from spicy_snow.data_acquisition.find_data import get_sentinel1_urls, find_snowcover_urls
from spicy_snow.utils.download import download_urls
from spicy_snow.processing.generate_dataarrays import generate_sentinel1_dataarray, generate_snowcover_dataarray, generate_forest_fraction_dataarray, convert_snowcover_dates_to_s1_overpasses

import pandas as pd
from spicy_snow.utils.raster import tif_to_dataarray, combine_close_images, da_to01
from spicy_snow.utils.checks import validate_aoi, within_conus

import xarray as xr

In [2]:
test_start_date = '2020-01-11'
test_stop_date = '2020-01-18'
test_aoi = [-120, 40.5, -119.5,41]

In [3]:
s1_urls = get_sentinel1_urls(test_start_date, test_stop_date, test_aoi)
s1_fps = download_urls(s1_urls, '/Users/zmhoppinen/Documents/spicy-snow/local/opera')
ds = xr.Dataset()
ds['vv'] = generate_sentinel1_dataarray(s1_fps, test_aoi, pol = 'VV')
spatial_reference = ds['vv'].isel(time =0)
ds['vh'] = generate_sentinel1_dataarray(s1_fps, test_aoi, pol = 'VH', ref = spatial_reference)

In [4]:
snowcover_urls = find_snowcover_urls(date_list = ds.time.dt.date, aoi = test_aoi)
snowcover_fps = download_urls(snowcover_urls, '/Users/zmhoppinen/Documents/spicy-snow/local/snowcover')

In [5]:
viirs_snowcover = generate_snowcover_dataarray(snowcover_fps, ref = spatial_reference)
viirs_snowcover = convert_snowcover_dates_to_s1_overpasses(viirs_snowcover, ds['vv'])
# table 1 https://nsidc.org/sites/default/files/documents/user-guide/multi_vnp10a1f-v002-userguide.pdf
ds['watermask'] = viirs_snowcover == 237
ds['snowcover'] = viirs_snowcover.where(viirs_snowcover <= 100)
ds['fcf'] = generate_forest_fraction_dataarray(test_aoi, ref = spatial_reference)

/Users/zmhoppinen/miniforge3/envs/spicy_testing/lib/python3.13/site-packages/pygeoutils/pygeoutils.py:300: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge(
